# PHÂN TÍCH TÀI CHÍNH

## Cài đặt thư viện

In [11]:
!pip install pandas pyarrow
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

## EDA

### Tổng quan

In [ ]:
# Load dữ liệu
df = pd.read_parquet('facts.parquet')

# === 1. Thông tin tổng quát ===
print("Shape của dataset:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nInfo:")
print(df.info())
print("\nMô tả thống kê:")
print(df.describe(include='all'))

# Kiểm tra missing values
print("\nMissing values (%):")
print((df.isnull().sum() / len(df) * 100).round(2))

Shape của dataset: (78474497, 5)

Columns: ['adsh', 'tag', 'start', 'end', 'value']

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78474497 entries, 0 to 78474496
Data columns (total 5 columns):
 #   Column  Dtype         
---  ------  -----         
 0   adsh    int64         
 1   tag     category      
 2   start   datetime64[ms]
 3   end     datetime64[ms]
 4   value   float64       
dtypes: category(1), datetime64[ms](2), float64(1), int64(1)
memory usage: 2.5 GB
None

Mô tả thống kê:
                adsh            tag                       start  \
count   7.847450e+07       78474497                    78474497   
unique           NaN           9638                         NaN   
top              NaN  NetIncomeLoss                         NaN   
freq             NaN        1047183                         NaN   
mean    1.187818e+14            NaN  2018-07-23 20:34:59.074000   
min     2.178110e+11            NaN         1900-01-01 00:00:00   
25%     1.000623e+14      

### Lọc dữ liệu cần thiết và tạo file mới

In [13]:
print("Tổng số loại tag (unique tags):", df['tag'].nunique())

# Xem top 40 tag phổ biến nhất
print("\n=== Top 40 tag phổ biến nhất ===")
print(df['tag'].value_counts().head(40))


Tổng số loại tag (unique tags): 9638

=== Top 40 tag phổ biến nhất ===
tag
NetIncomeLoss                                                                                  1047183
StockholdersEquity                                                                              870746
OperatingIncomeLoss                                                                             751511
CashAndCashEquivalentsAtCarryingValue                                                           724388
IncomeTaxExpenseBenefit                                                                         674161
WeightedAverageNumberOfSharesOutstandingBasic                                                   653146
WeightedAverageNumberOfDilutedSharesOutstanding                                                 627913
Assets                                                                                          565124
LiabilitiesAndStockholdersEquity                                                                54655

In [14]:
import pandas as pd

# Load file gốc
df = pd.read_parquet('facts.parquet')

# Lấy Top 40 tag phổ biến nhất
top40_tags = df['tag'].value_counts().head(40).index.tolist()

print("=== Top 40 tags được chọn ===")
for i, tag in enumerate(top40_tags, 1):
    count = df['tag'].value_counts()[tag]
    print(f"{i:2d}. {tag} ({count:,} records)")

# Lọc dữ liệu
df_filtered = df[df['tag'].isin(top40_tags)].copy()

# Thống kê
print("\n" + "="*60)
print("Kích thước trước khi lọc :", df.shape)
print("Kích thước sau khi lọc  :", df_filtered.shape)
print("Tỷ lệ giữ lại          :", round(len(df_filtered)/len(df)*100, 2), "%")
print("Số tag trong file mới  :", df_filtered['tag'].nunique())

# Lưu file mới (nén gzip để tiết kiệm dung lượng)
df_filtered.to_parquet('facts_top40_tags.parquet', 
                       index=False, 
                       compression='gzip')

print("\n✅ Đã lưu file mới: **facts_top40_tags.parquet**")

=== Top 40 tags được chọn ===
 1. NetIncomeLoss (1,047,183 records)
 2. StockholdersEquity (870,746 records)
 3. OperatingIncomeLoss (751,511 records)
 4. CashAndCashEquivalentsAtCarryingValue (724,388 records)
 5. IncomeTaxExpenseBenefit (674,161 records)
 6. WeightedAverageNumberOfSharesOutstandingBasic (653,146 records)
 7. WeightedAverageNumberOfDilutedSharesOutstanding (627,913 records)
 8. Assets (565,124 records)
 9. LiabilitiesAndStockholdersEquity (546,559 records)
10. RetainedEarningsAccumulatedDeficit (522,824 records)
11. ComprehensiveIncomeNetOfTax (520,744 records)
12. NetCashProvidedByUsedInOperatingActivities (517,683 records)
13. NetCashProvidedByUsedInFinancingActivities (503,080 records)
14. InterestExpense (492,576 records)
15. NetCashProvidedByUsedInInvestingActivities (466,088 records)
16. CommonStockSharesAuthorized (460,583 records)
17. IncomeLossFromContinuingOperationsBeforeIncomeTaxesExtraordinaryItemsNoncontrollingInterest (459,111 records)
18. CommonStockVa